# Exploração — NYC Yellow Taxi
Objetivo: decidir recorte, colunas e regras de validação para o ingest.

In [14]:
import duckdb, requests, pathlib

con = duckdb.connect()
raw = pathlib.Path("../data/raw")
base = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{}-{:02d}.parquet"

## 1.1 Volume sem baixar

In [15]:
%%time
urls = [base.format(a, m) for a in range(2019, 2026) for m in range(1, 13)]

total = sum(int(requests.head(u).headers.get("Content-Length", 0)) for u in urls)
print(f"Tamanho: {total/1e9:.2f} GB em {len(urls)} arquivos")

linhas = con.sql(f"SELECT sum(num_rows) FROM parquet_file_metadata({urls})").fetchone()[0]
print(f"Linhas: {linhas:,}")

Tamanho: 4.87 GB em 84 arquivos
Linhas: 308,010,490
CPU times: total: 16.2 s
Wall time: 25.9 s


## 1.2 Schema 2019 vs 2025

In [16]:
for ano in (2019, 2025):
    print(f"===== {ano} =====")
    con.sql(f"DESCRIBE SELECT * FROM '{base.format(ano, 1)}'").show()

===== 2019 =====
┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ VendorID              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_pickup_datetime  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_dropoff_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_distance         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ RatecodeID            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_and_fwd_flag    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ DOLocationID         

## 1.3 Um mês em detalhe

In [17]:
arquivos = {
    "yellow_tripdata_2024-01.parquet": base.format(2024, 1),
    "taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
}
for nome, url in arquivos.items():
    destino = raw / nome
    if not destino.exists():
        destino.write_bytes(requests.get(url).content)
    print(nome, f"{destino.stat().st_size/1e6:.1f} MB")

yellow_tripdata_2024-01.parquet 50.0 MB
taxi_zone_lookup.csv 0.0 MB


In [18]:
f = "../data/raw/yellow_tripdata_2024-01.parquet"
con.sql(f"""
  SELECT count(*) FILTER (WHERE tpep_pickup_datetime <  '2024-01-01'
                            OR  tpep_pickup_datetime >= '2024-02-01') AS fora_do_mes,
         min(tpep_pickup_datetime), max(tpep_pickup_datetime)
  FROM '{f}'
""").show()

┌─────────────┬───────────────────────────┬───────────────────────────┐
│ fora_do_mes │ min(tpep_pickup_datetime) │ max(tpep_pickup_datetime) │
│    int64    │         timestamp         │         timestamp         │
├─────────────┼───────────────────────────┼───────────────────────────┤
│          18 │ 2002-12-31 22:59:39       │ 2024-02-01 00:01:15       │
└─────────────┴───────────────────────────┴───────────────────────────┘



In [19]:
con.sql(f"SUMMARIZE SELECT * FROM '{f}'").show()

┌───────────────────────┬─────────────┬─────────────────────┬─────────────────────┬───────────────┬────────────────────────────┬─────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬─────────┬─────────────────┐
│      column_name      │ column_type │         min         │         max         │ approx_unique │            avg             │         std         │            q25             │            q50             │            q75             │  count  │ null_percentage │
│        varchar        │   varchar   │       varchar       │       varchar       │     int64     │          varchar           │       varchar       │          varchar           │          varchar           │          varchar           │  int64  │  decimal(9,2)   │
├───────────────────────┼─────────────┼─────────────────────┼─────────────────────┼───────────────┼────────────────────────────┼─────────────────────┼────────────────────────────┼───────────────────────

In [20]:
con.sql("SELECT * FROM '../data/raw/taxi_zone_lookup.csv' LIMIT 10").show()

┌────────────┬───────────────┬─────────────────────────┬──────────────┐
│ LocationID │    Borough    │          Zone           │ service_zone │
│   int64    │    varchar    │         varchar         │   varchar    │
├────────────┼───────────────┼─────────────────────────┼──────────────┤
│          1 │ EWR           │ Newark Airport          │ EWR          │
│          2 │ Queens        │ Jamaica Bay             │ Boro Zone    │
│          3 │ Bronx         │ Allerton/Pelham Gardens │ Boro Zone    │
│          4 │ Manhattan     │ Alphabet City           │ Yellow Zone  │
│          5 │ Staten Island │ Arden Heights           │ Boro Zone    │
│          6 │ Staten Island │ Arrochar/Fort Wadsworth │ Boro Zone    │
│          7 │ Queens        │ Astoria                 │ Boro Zone    │
│          8 │ Queens        │ Astoria Park            │ Boro Zone    │
│          9 │ Queens        │ Auburndale              │ Boro Zone    │
│         10 │ Queens        │ Baisley Park            │ Boro Zo

## 1.4 Pendências

In [21]:
con.sql(f"""
  SELECT name, type, count(DISTINCT file_name) AS arquivos
  FROM parquet_schema({urls})
  WHERE num_children IS NULL
  GROUP BY ALL
  ORDER BY name, arquivos DESC
""").show(max_rows=100)

┌───────────────────────┬────────────┬──────────┐
│         name          │    type    │ arquivos │
│        varchar        │  varchar   │  int64   │
├───────────────────────┼────────────┼──────────┤
│ Airport_fee           │ DOUBLE     │       35 │
│ DOLocationID          │ INT64      │       49 │
│ DOLocationID          │ INT32      │       35 │
│ PULocationID          │ INT64      │       49 │
│ PULocationID          │ INT32      │       35 │
│ RatecodeID            │ DOUBLE     │       49 │
│ RatecodeID            │ INT64      │       35 │
│ VendorID              │ INT64      │       49 │
│ VendorID              │ INT32      │       35 │
│ airport_fee           │ DOUBLE     │       29 │
│ airport_fee           │ INT32      │       20 │
│ cbd_congestion_fee    │ DOUBLE     │       12 │
│ congestion_surcharge  │ DOUBLE     │       84 │
│ extra                 │ DOUBLE     │       84 │
│ fare_amount           │ DOUBLE     │       84 │
│ improvement_surcharge │ DOUBLE     │       84 │


In [22]:
con.sql(f"""
  SELECT payment_type, count(*) AS n,
         count(*) FILTER (WHERE passenger_count IS NULL) AS nulos
  FROM '{f}' GROUP BY 1 ORDER BY 1
""").show()

┌──────────────┬─────────┬────────┐
│ payment_type │    n    │ nulos  │
│    int64     │  int64  │ int64  │
├──────────────┼─────────┼────────┤
│            0 │  140162 │ 140162 │
│            1 │ 2319046 │      0 │
│            2 │  439191 │      0 │
│            3 │   19597 │      0 │
│            4 │   46628 │      0 │
└──────────────┴─────────┴────────┘



In [23]:
con.sql("SELECT * FROM '../data/raw/taxi_zone_lookup.csv' WHERE LocationID >= 263").show()

┌────────────┬───────────┬────────────────┬──────────────┐
│ LocationID │  Borough  │      Zone      │ service_zone │
│   int64    │  varchar  │    varchar     │   varchar    │
├────────────┼───────────┼────────────────┼──────────────┤
│        263 │ Manhattan │ Yorkville West │ Yellow Zone  │
│        264 │ Unknown   │ N/A            │ N/A          │
│        265 │ N/A       │ Outside of NYC │ N/A          │
└────────────┴───────────┴────────────────┴──────────────┘



In [24]:
con.sql(f"""
  SELECT type, min(file_name) AS primeiro, max(file_name) AS ultimo
  FROM parquet_schema({urls})
  WHERE name = 'passenger_count'
  GROUP BY type
""").show()

┌─────────┬─────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────┐
│  type   │                                    primeiro                                     │                                     ultimo                                      │
│ varchar │                                     varchar                                     │                                     varchar                                     │
├─────────┼─────────────────────────────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────────────────────┤
│ DOUBLE  │ https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet │ https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet │
│ INT64   │ https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet │ https://d37ci6vzurychx.clo

In [25]:
con.sql(f"""
  SELECT payment_type, count(*) AS negativos
  FROM '{f}' WHERE total_amount < 0
  GROUP BY 1 ORDER BY 1
""").show()

┌──────────────┬───────────┐
│ payment_type │ negativos │
│    int64     │   int64   │
├──────────────┼───────────┤
│            0 │         2 │
│            1 │        29 │
│            2 │      8326 │
│            3 │      5741 │
│            4 │     21406 │
└──────────────┴───────────┘



In [26]:
con.sql("""
  SELECT count(DISTINCT row_group_id) AS row_groups,
         max(row_group_num_rows)      AS linhas_por_bloco
  FROM parquet_metadata('../data/lake/trips/ano=2019/mes=1/trips.parquet')
""").show()

┌────────────┬──────────────────┐
│ row_groups │ linhas_por_bloco │
│   int64    │      int64       │
├────────────┼──────────────────┤
│          4 │          2000896 │
└────────────┴──────────────────┘



# Exploração — NYC Yellow Taxi
Objetivo: decidir recorte, colunas e regras de validação para o ingest.

Fonte de referência: *Data Dictionary – Yellow Taxi Trip Records*, NYC TLC, 18/03/2025.

---

## Volume (jan/2019 – dez/2025)
- **Tamanho:** 4,87 GB em 84 arquivos Parquet
- **Linhas:** 308.010.490
- **Tempo:** 1min37s (wall) / 40,7 s (CPU)
- **Como:** sem baixar os arquivos. `requests.head` lê só o tamanho (`Content-Length`) e `parquet_file_metadata` lê só o rodapé de cada Parquet.
- **Obs.:** a maior parte do tempo vem das 84 requisições HEAD feitas em sequência.

---

## Schema (84 arquivos, via `parquet_schema`)

**Ponto de corte em fev/2023.** Várias colunas mudam de tipo ao mesmo tempo:

| Coluna | 2019-01 → 2023-01 (49 arquivos) | 2023-02 → 2025-12 (35 arquivos) |
|---|---|---|
| `passenger_count` | DOUBLE | INT64 |
| `RatecodeID` | DOUBLE | INT64 |
| `VendorID` | INT64 | INT32 |
| `PULocationID` / `DOLocationID` | INT64 | INT32 |

- **Validação do corte:** os intervalos de meses (49 e 35) batem com as contagens de arquivos por tipo, então o corte é limpo, sem alternância.
- **`airport_fee` com dois nomes:** `airport_fee` (49 arquivos, DOUBLE ou INT32) e `Airport_fee` (35 arquivos, DOUBLE).
- **`cbd_congestion_fee`:** só nos 12 arquivos de 2025 (cobrança iniciada em 05/01/2025, segundo o dicionário).
- **Timestamps:** o tipo físico INT64 é o armazenamento normal do Parquet. O DuckDB lê como TIMESTAMP.
- **Consequência para o ingest:** padronizar nomes para minúsculo e forçar um tipo único por coluna.

---

## Qualidade (amostra: jan/2024, 2.964.624 linhas)

### Datas fora do mês
- 18 corridas com `tpep_pickup_datetime` fora de jan/2024
- Mínimo: 2002-12-31 22:59:39 · Máximo: 2024-02-01 00:01:15

### Nulos (confirmado)
- 5 colunas com 4,73% de nulos: `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, `Airport_fee`
- Os 140.162 nulos estão **todos** em `payment_type = 0`, que o dicionário define como **Flex Fare trip**
- Não é sujeira: é uma categoria de corrida que não registra essas colunas

### Códigos categóricos (conferidos no dicionário)
| Coluna | Observado (jan/2024) | Códigos válidos | Conclusão |
|---|---|---|---|
| `VendorID` | 1 a 6 | 1, 2, 6, 7 | Válido (6 = Myle Technologies) |
| `RatecodeID` | 1 a 99 | 1–6 e 99 | Válido (99 = Null/unknown, código oficial) |
| `payment_type` | 0 a 4 | 0–6 | Válido (0 = Flex Fare; 5 e 6 não aparecem na amostra) |

### Valores suspeitos ou impossíveis
| Coluna | Mín | Máx | Observação |
|---|---|---|---|
| `trip_distance` | 0,0 | 312.722,3 | Distância zero e máximo impossível |
| `passenger_count` | 0 | 9 | Zero passageiros |
| `fare_amount` | -899,0 | 5.000,0 | Negativos e máximo extremo |
| `total_amount` | -900,0 | 5.000,0 | Negativos e máximo extremo |

### Valores monetários negativos
- Presentes em: `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `Airport_fee`
- **Hipótese:** estornos e ajustes. O dicionário tem os códigos `4 = Dispute` e `6 = Voided trip`, que podem explicar parte deles.
- **Pendente:** cruzar valores negativos × `payment_type`.

---

## Dimensão de zonas (`taxi_zone_lookup.csv`)
- 4 colunas: `LocationID`, `Borough`, `Zone`, `service_zone`
- Chave de JOIN: `LocationID` ↔ `PULocationID` / `DOLocationID`
- **Zonas especiais:** 264 = Unknown · 265 = Outside of NYC
- Manter como categorias próprias nas views

---

## Alertas analíticos (do dicionário)
- **`tip_amount`** registra só gorjetas no cartão. Gorjetas em dinheiro não entram, então "gorjeta em dinheiro ≈ 0" é uma limitação do dado, não um comportamento.
- **`total_amount`** também não inclui gorjetas em dinheiro.
- **`airport_fee`** só se aplica a embarques em LaGuardia e JFK.

---

## Pendências
- [ ] Valores negativos × `payment_type`
- [ ] Proposta: recorte, colunas e regras

---

## Proposta para o ingest

### Recorte
- **Período:** jan/2019 – dez/2025 (84 arquivos, ~308 milhões de linhas, 4,87 GB)
- **Justificativa:** volume suficiente para demonstrar o contraste "lake grande vs bytes lidos por query"; inclui a mudança de schema de 2023 e a taxa de 2025, que ilustram tratamento de schema drift

### Colunas mantidas
| Coluna | Pergunta que responde |
|---|---|
| `tpep_pickup_datetime` | Quando a corrida começou? (base da partição) |
| `tpep_dropoff_datetime` | Quanto tempo durou? |
| `pu_location_id` / `do_location_id` | De onde para onde? (JOIN com zonas) |
| `passenger_count` | Quantas pessoas? |
| `trip_distance` | Qual a distância? |
| `ratecode_id` | Foi tarifa de aeroporto, negociada...? |
| `payment_type` | Como pagou? |
| `fare_amount` | Quanto custou a tarifa base? |
| `tip_amount` | Quanto de gorjeta (só cartão)? |
| `tolls_amount` | Quanto de pedágio? |
| `total_amount` | Quanto custou no total? |
| `congestion_surcharge` | Quanto de taxa de congestionamento estadual? |
| `airport_fee` | Quanto de taxa de aeroporto? |
| `cbd_congestion_fee` | Qual o impacto da taxa de Manhattan em 2025? |

**Descartadas:** `VendorID`, `store_and_fwd_flag`, `extra`, `mta_tax`, `improvement_surcharge`. Nenhuma das views planejadas usa essas colunas.

### Regras
| Problema | Decisão | Justificativa |
|---|---|---|
| Datas fora do mês do arquivo | **Descartar** | Impossíveis; criariam partições absurdas (ex.: `ano=2002`) |
| Flex Fare (`payment_type = 0`, nulos) | **Manter** | Corridas reais; descartar subestima o volume |
| `trip_distance` ≤ 0 ou > 100 milhas | **Sinalizar** (`is_suspeita`) | Pode ser cancelamento ou erro de taxímetro; as views filtram por padrão, mas o dado não some |
| `total_amount` > US$ 1.000 | **Sinalizar** (`is_suspeita`) | Mesmo critério: extremo, mas não comprovadamente falso |
| Valores negativos | **Manter** | Concentrados em Dispute (4); remover inflaria a receita |
| `passenger_count = 0` | **Converter para NULL** | Zero passageiros não existe; NULL comunica "desconhecido" |
| Mudanças de schema | **Padronizar** | Nomes em snake_case minúsculo; tipo único por coluna (IDs e contagens INTEGER, valores DOUBLE) |
| `airport_fee` com dois nomes | **Unificar** | Mesma informação; virar uma coluna só |
| `cbd_congestion_fee` antes de 2025 | **NULL** | A taxa não existia; NULL diz "não se aplica", 0 diria "cobrado zero" |
| Zonas 264/265 | **Manter** | Viram categorias "Desconhecida" e "Fora de NYC" nas views |